In [ ]:
!pip install -qq catboost

     |████████████████████████████████| 66.1MB 46kB/s 


In [ ]:
# General 
import pandas as pd
import numpy as np
import requests
from io import StringIO 
import datetime as dt
import re
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRFClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.metrics import log_loss
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from fastai.tabular import add_cyclic_datepart, add_datepart
from catboost import CatBoostRegressor, CatBoostClassifier
from tqdm import tqdm, tqdm_notebook
from itertools import combinations

## Features
 - Date features
 - Age
 - Ratio, of age and product sum
 - Frequency of categorical variables
 - Category encoding
 - Dummy encoding
 - Combining categorical variables
 - Ratios
 - Psudo labelling
 - Remove outliers
 - categorical encoding to product sum
 - use high cross validation folds -20

In [ ]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
sample = pd.read_csv('SampleSubmission.csv')

In [ ]:
train.head()

,ID,join_date,sex,marital_status,birth_year,branch_code,occupation_code,occupation_category_code,P5DA,RIBP,8NN1,7POT,66FJ,GYSR,SOP4,RVSZ,PYUQ,LJR9,N2MW,AHXO,BSTQ,FM3X,K6QO,QBOL,JWFN,JZ9D,J9JW,GHYX,ECY3
0,4WKQSBB,1/2/2019,F,M,1987,1X1H,2A7I,T4MS,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
1,CP5S02H,1/6/2019,F,M,1981,UAOD,2A7I,T4MS,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
2,2YKDILJ,1/6/2013,M,U,1991,748L,QZYX,90QI,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1
3,2S9E81J,1/8/2019,M,M,1990,1X1H,BP09,56SI,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
4,BHDYVFT,1/8/2019,M,M,1990,748L,NO3L,T4MS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0


In [ ]:
test.head()

,ID,join_date,sex,marital_status,birth_year,branch_code,occupation_code,occupation_category_code,P5DA,RIBP,8NN1,7POT,66FJ,GYSR,SOP4,RVSZ,PYUQ,LJR9,N2MW,AHXO,BSTQ,FM3X,K6QO,QBOL,JWFN,JZ9D,J9JW,GHYX,ECY3
0,F86J5PC,1/12/2018,M,M,1984,94KC,DZRV,90QI,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,H6141K3,1/10/2019,M,M,1996,1X1H,J9SY,90QI,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
2,RBAYUXZ,1/1/2020,F,W,1968,UAOD,2A7I,T4MS,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
3,KCBILBQ,1/2/2019,M,M,1989,94KC,2A7I,T4MS,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,LSEC1ZJ,1/2/2020,F,M,1982,UAOD,0KID,T4MS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [ ]:
sample.head()

,ID X PCODE,Label
0,F86J5PC X P5DA,0
1,F86J5PC X RIBP,0
2,F86J5PC X 8NN1,0
3,F86J5PC X 7POT,0
4,F86J5PC X 66FJ,0


In [ ]:
products = train.columns[8:].tolist()
train['product_sum'], test['product_sum']  = train[products].sum(axis = 1), [x+1 for x in test[products].sum(axis = 1)]
main_train = train.copy()
train.shape, test.shape, sample.shape

((29132, 30), (10000, 30), (210000, 2))

In [ ]:
target, body = [], []
for i in tqdm_notebook(range(train.shape[0])):
  row = train.iloc[i, :]
  for product in products:
    if row[product] == 1:
      stripped = row.copy()
      stripped.loc[product] = 0
      target.append(product)
      body.append(stripped.values)

train_stripped = pd.DataFrame(body, columns = train.columns)
train_stripped['target'] = target
train_stripped.shape, main_train.shape, train.shape

((66353, 31), (29132, 30), (29132, 30))

In [ ]:
train_stripped.head()

,ID,join_date,sex,marital_status,birth_year,branch_code,occupation_code,occupation_category_code,P5DA,RIBP,8NN1,7POT,66FJ,GYSR,SOP4,RVSZ,PYUQ,LJR9,N2MW,AHXO,BSTQ,FM3X,K6QO,QBOL,JWFN,JZ9D,J9JW,GHYX,ECY3,product_sum,target
0,4WKQSBB,1/2/2019,F,M,1987,1X1H,2A7I,T4MS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2,RVSZ
1,4WKQSBB,1/2/2019,F,M,1987,1X1H,2A7I,T4MS,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,K6QO
2,CP5S02H,1/6/2019,F,M,1981,UAOD,2A7I,T4MS,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2,RVSZ
3,CP5S02H,1/6/2019,F,M,1981,UAOD,2A7I,T4MS,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2,K6QO
4,2YKDILJ,1/6/2013,M,U,1991,748L,QZYX,90QI,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,3,SOP4


In [ ]:
target = train_stripped.target

train_stripped['separator'] = 0
test['separator'] = 1

train, test = train_stripped.align(test, join = 'inner', axis = 1)

comb = pd.concat([train, test])

In [ ]:
# Function to add date features
def add_datepart(df, fldname, drop=True):
    fld = df[fldname]
    if not np.issubdtype(fld.dtype, np.datetime64):
        df[fldname] = fld = pd.to_datetime(fld, infer_datetime_format=True)
    targ_pre = re.sub('[Dd]ate$', '', fldname)
    for n in ('Year', 'Month', 'Week', 'Day', 'Dayofweek', 'Dayofyear',
           'Is_month_end', 'Is_month_start', 'Is_quarter_end', 'Is_quarter_start', 'Is_year_end', 'Is_year_start'):
      
# 'Is_month_end', 'Is_month_start', 'Is_quarter_end', 'Is_quarter_start', 'Is_year_end', 'Is_year_start'
        df[targ_pre+n] = getattr(fld.dt,n.lower())
    df[targ_pre+'Elapsed'] = fld.astype(np.int64) // 10**9
    if drop: df.drop(fldname, axis=1, inplace=True)

In [ ]:
# Fill missing date with mode
comb.fillna('2018-01-05 00:00:00', inplace = True)
comb.join_date = pd.to_datetime(comb.join_date)

# Adding date features
add_datepart(comb, 'join_date', False)

comb['duration'] = (comb.join_date - comb.join_date.min()).dt.days

# Adding cyclic date features
add_cyclic_datepart(comb, 'join_date')

# Calculate age column
comb['age'] = comb.join_Year - comb.birth_year

In [ ]:
comb.marital_status = comb.marital_status.apply(lambda x: x.replace('f', 'F'))

In [ ]:
cat_cols = ['sex',	'marital_status',	'branch_code',	'occupation_code',	'occupation_category_code']
for col in cat_cols:
  comb[col] = comb[col].apply(lambda x: x.strip())

In [ ]:
rare_occupation = comb.occupation_code.value_counts()[comb.occupation_code.value_counts() <= 24].index.tolist()
occupation_code = []
for i in comb.occupation_code:
  if i in rare_occupation: occupation_code.append('rare_occupation')
  else: occupation_code.append(i)

comb['occupation_code'] = occupation_code

In [ ]:
rare_branch_code = comb.branch_code.value_counts()[comb.branch_code.value_counts() <= 32].index.tolist()
branch_code = []
for i in comb.branch_code:
  if i in rare_branch_code: branch_code.append('rare_branch_code')
  else: branch_code.append(i)

comb['branch_code'] = branch_code

In [ ]:
comb['product_sum_age_ratio'] =  comb.age / comb.product_sum
comb['product_sum_duration_ratio'] =  comb.duration / comb.product_sum

In [ ]:
for col in cat_cols:
  comb[col+'_freq'] = comb[col].map(train_stripped[col].value_counts().to_dict())

for col in [x for x in comb.columns if 'freq' in x]:
  comb[col+'_pdct_sum_ratio'] = comb[col] / comb.product_sum

In [ ]:
for col in tqdm_notebook(cat_cols):
  for product in products:
    for ag in ['mean', 'std', 'var', 'max', 'median', 'sum', 'sem', 'min']:
      comb[col+'_'+product+'_'+ag] = comb[col].map(comb.groupby(col)[product].agg(ag))

In [ ]:
for combs in tqdm_notebook(list(combinations(cat_cols, 2))):
  comb[combs[0]+'_'+combs[1]+'_nunique'] = comb[combs[0]].map(comb.groupby(combs[0])[combs[1]].nunique())

In [ ]:
aggs = ['mean', 'std', 'var', 'max', 'median', 'sum', 'sem', 'min']
for agg in tqdm_notebook(aggs):
  for col in products:
    for i in ['birth_year', 'join_Elapsed', 'age',	'product_sum_age_ratio', 'product_sum', 'product_sum_duration_ratio']:
      comb[col+'_'+i+'_'+agg] = comb[col].map(comb.groupby(col)[i].agg(agg))

In [ ]:
comb['pattern'] = ''
for product in tqdm_notebook(products): comb.pattern = comb.pattern + comb[product].astype('str')
comb.pattern = pd.factorize(comb.pattern)[0]
comb['pattern_freq'] = comb.pattern.map(comb.pattern.value_counts(normalize = True))
comb.loc[comb.pattern.isin(comb.pattern.value_counts()[comb.pattern.value_counts() < 10].index.tolist()), 'pattern'] = np.nan

In [ ]:
comb.shape

(76353, 1925)

In [ ]:
non_unique_cols = comb.columns[comb.nunique() <= 1].to_list()
comb = comb[comb.columns.difference(non_unique_cols)]

In [ ]:
len(non_unique_cols), comb.shape

(354, (76353, 1571))

In [ ]:
comb = pd.get_dummies(comb, columns = ['sex',	'marital_status',	'branch_code',	'occupation_category_code', 'occupation_code'])
comb.head()

,66FJ,66FJ_age_max,66FJ_age_mean,66FJ_age_median,66FJ_age_min,66FJ_age_sem,66FJ_age_std,66FJ_age_sum,66FJ_age_var,66FJ_birth_year_max,66FJ_birth_year_mean,66FJ_birth_year_median,66FJ_birth_year_min,66FJ_birth_year_sem,66FJ_birth_year_std,66FJ_birth_year_sum,66FJ_birth_year_var,66FJ_join_Elapsed_max,66FJ_join_Elapsed_mean,66FJ_join_Elapsed_median,66FJ_join_Elapsed_min,66FJ_join_Elapsed_sem,66FJ_join_Elapsed_std,66FJ_join_Elapsed_sum,66FJ_join_Elapsed_var,66FJ_product_sum_age_ratio_max,66FJ_product_sum_age_ratio_mean,66FJ_product_sum_age_ratio_median,66FJ_product_sum_age_ratio_min,66FJ_product_sum_age_ratio_sem,66FJ_product_sum_age_ratio_std,66FJ_product_sum_age_ratio_sum,66FJ_product_sum_age_ratio_var,66FJ_product_sum_duration_ratio_mean,66FJ_product_sum_duration_ratio_median,66FJ_product_sum_duration_ratio_min,66FJ_product_sum_duration_ratio_sem,66FJ_product_sum_duration_ratio_std,66FJ_product_sum_duration_ratio_sum,66FJ_product_sum_duration_ratio_var,...,occupation_code_NSJX,occupation_code_OPVX,occupation_code_OYQF,occupation_code_P4MD,occupation_code_PKW3,occupation_code_PMAI,occupation_code_Q57T,occupation_code_Q6J6,occupation_code_QJID,occupation_code_QZYX,occupation_code_R44Q,occupation_code_RE69,occupation_code_RUFT,occupation_code_RXV3,occupation_code_RY9B,occupation_code_S9KU,occupation_code_SF1X,occupation_code_SS6D,occupation_code_SST3,occupation_code_SSTX,occupation_code_T6AB,occupation_code_TUN1,occupation_code_U37O,occupation_code_U9RX,occupation_code_UBBX,occupation_code_UJ5T,occupation_code_VREH,occupation_code_WE0G,occupation_code_WE7U,occupation_code_WMTK,occupation_code_WSID,occupation_code_WV7U,occupation_code_XC1N,occupation_code_XHJD,occupation_code_XVMH,occupation_code_Y1WG,occupation_code_YMGT,occupation_code_ZCQR,occupation_code_ZKQ3,occupation_code_rare_occupation
0,0,87,38.406408,38,8,0.033481,9.215062,2909439,84.917362,2011,1979.329026,1980,1932,0.033717,9.279988,149942091,86.118174,1578441600,1.506873e+09,1515369600,1262649600,219411.56131,6.038962e+07,114151676544000,3.646906e+15,43.5,16.857521,16.5,2.642857,0.01881,5.177172,1.277025e+06,26.803112,1249.304132,1280.5,0.0,1.467868,404.007847,9.463979e+07,163222.34077,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,87,38.406408,38,8,0.033481,9.215062,2909439,84.917362,2011,1979.329026,1980,1932,0.033717,9.279988,149942091,86.118174,1578441600,1.506873e+09,1515369600,1262649600,219411.56131,6.038962e+07,114151676544000,3.646906e+15,43.5,16.857521,16.5,2.642857,0.01881,5.177172,1.277025e+06,26.803112,1249.304132,1280.5,0.0,1.467868,404.007847,9.463979e+07,163222.34077,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,87,38.406408,38,8,0.033481,9.215062,2909439,84.917362,2011,1979.329026,1980,1932,0.033717,9.279988,149942091,86.118174,1578441600,1.506873e+09,1515369600,1262649600,219411.56131,6.038962e+07,114151676544000,3.646906e+15,43.5,16.857521,16.5,2.642857,0.01881,5.177172,1.277025e+06,26.803112,1249.304132,1280.5,0.0,1.467868,404.007847,9.463979e+07,163222.34077,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,87,38.406408,38,8,0.033481,9.215062,2909439,84.917362,2011,1979.329026,1980,1932,0.033717,9.279988,149942091,86.118174,1578441600,1.506873e+09,1515369600,1262649600,219411.56131,6.038962e+07,114151676544000,3.646906e+15,43.5,16.857521,16.5,2.642857,0.01881,5.177172,1.277025e+06,26.803112,1249.304132,1280.5,0.0,1.467868,404.007847,9.463979e+07,163222.34077,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,87,38.406408,38,8,0.033481,9.215062,2909439,84.917362,2011,1979.329026,1980,1932,0.033717,9.279988,149942091,86.118174,1578441600,1.506873e+09,1515369600,1262649600,219411.56131,6.038962e+07,114151676544000,3.646906e+15,43.5,16.857521,16.5,2.642857,0.01881,5.177172,1.277025e+06,26.803112,1249.304132,1280.5,0.0,1.467868,404.007847,9.463979e+07,163222.34077,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0

In [ ]:
train = comb[comb.separator == 0]
test = comb[comb.separator == 1]

train.drop('separator', axis = 1, inplace = True)
test.drop('separator', axis = 1, inplace = True)

In [ ]:
train.shape, test.shape

((66353, 1699), (10000, 1699))

In [ ]:
train['target'] = target.copy()

In [ ]:
def submission_preparation(predictions):
  prob_df, test_sub = pd.DataFrame(np.mean(predictions, 0)), test.copy()
  for a in tqdm_notebook(range(test.shape[0])):
    for b,c in enumerate(list(cat.classes_)):
      if test_sub.loc[a,c]!=1:test_sub.loc[a,c]=prob_df.loc[a, b]
  sub_df = test_sub[['ID']+list(cat.classes_)]

  sub_df.set_index('ID', inplace = True)
  sub_df = pd.DataFrame({'ID X PCODE': [' X '.join(x) for x in sub_df.stack().index.to_flat_index()], 'Label': sub_df.stack()}).reset_index(drop = True)
  return sub_df

In [ ]:
%%time
X = train.copy()
X.reset_index(drop = True, inplace = True)
y = X.target
X = X.drop('target', axis = 1)

tes = test.drop('ID', axis = 1)

cat_stratified_preds = []
y_preds = []
y_preds_index = []
y_trues = []
kfolds = StratifiedKFold(n_splits = 15, random_state = 1, shuffle=True)
for train_index, test_index in kfolds.split(X, y, X.ID):
  X_train, X_test, y_train, y_test = X.loc[train_index], X.loc[test_index], y[train_index], y[test_index]
  X_train, X_test = X_train.drop('ID',axis = 1), X_test.drop('ID', axis = 1)
  cat = CatBoostClassifier(n_estimators=10000, learning_rate=0.1, eval_metric='MultiClass', verbose=100, task_type = 'GPU')
  cat.fit(X_train, y_train,
            eval_set=[(X_test, y_test)],
            early_stopping_rounds=100,
            use_best_model = True)
  cat_stratified_preds.append(cat.predict_proba(tes))
  y_pred = cat.predict_proba(X_test)
  y_preds.extend(y_pred)
  y_trues.extend(y_test)
  y_preds_index.append(test_index)
print(f'OOOOOOOOOOOOOOOOOOOOOO____{log_loss(y_trues, y_preds)}_____OOOOOOOOOOOOOOOO')

cat_stratified_sub = submission_preparation(cat_stratified_preds)

0:	learn: 1.7728113	test: 1.7812483	best: 1.7812483 (0)	total: 30.4ms	remaining: 5m 3s
100:	learn: 0.3940832	test: 0.4033770	best: 0.4033770 (100)	total: 2.45s	remaining: 4m
200:	learn: 0.3494973	test: 0.3743564	best: 0.3743564 (200)	total: 4.71s	remaining: 3m 49s
300:	learn: 0.3270829	test: 0.3638246	best: 0.3638246 (300)	total: 6.87s	remaining: 3m 41s
400:	learn: 0.3122610	test: 0.3584574	best: 0.3584509 (399)	total: 8.94s	remaining: 3m 34s
500:	learn: 0.2984365	test: 0.3550597	best: 0.3550597 (500)	total: 11.1s	remaining: 3m 30s
600:	learn: 0.2867497	test: 0.3522791	best: 0.3522520 (599)	total: 13.2s	remaining: 3m 25s
700:	learn: 0.2761462	test: 0.3504167	best: 0.3501650 (691)	total: 15.3s	remaining: 3m 23s
800:	learn: 0.2666943	test: 0.3487308	best: 0.3487308 (800)	total: 17.5s	remaining: 3m 20s
900:	learn: 0.2577779	test: 0.3483461	best: 0.3482671 (898)	total: 19.6s	remaining: 3m 18s
1000:	learn: 0.2496456	test: 0.3475035	best: 0.3474939 (995)	total: 21.8s	remaining: 3m 15s
1100:	


CPU times: user 9min 22s, sys: 5min 9s, total: 14min 31s
Wall time: 8min 37s


In [ ]:
%%time
X = train.copy()
X.reset_index(drop = True, inplace = True)
y = X.target
X = X.drop('target', axis = 1)
tes = test.drop('ID', axis = 1)

lgbm_grp_preds = []
y_preds = []
y_preds_index = []
y_trues = []
kfolds = GroupKFold(n_splits = 15)
for train_index, test_index in kfolds.split(X, y, X.ID):
  X_train, X_test, y_train, y_test = X.loc[train_index], X.loc[test_index], y[train_index], y[test_index]
  X_train, X_test = X_train.drop('ID',axis = 1), X_test.drop('ID', axis = 1)
  cat = LGBMClassifier(n_estimators = 10000, learning_rate = 0.05, colsample_bytree = 0.2, reg_alpha=2, reg_lambda=3)
  cat.fit(X_train, y_train,
            eval_set=[(X_train, y_train), (X_test, y_test)],
            early_stopping_rounds=100,
            verbose = 100)
  lgbm_grp_preds.append(cat.predict_proba(tes))
  y_pred = cat.predict_proba(X_test)
  y_preds.extend(y_pred)
  y_trues.extend(y_test)
  y_preds_index.append(test_index)
print(f'OOOOOOOOOOOOOOOOOOOOOO____{log_loss(y_trues, y_preds)}_____OOOOOOOOOOOOOOOO')

lgbm_grp_sub = submission_preparation(lgbm_grp_preds)

Training until validation scores don't improve for 100 rounds.
[100]	training's multi_logloss: 0.333201	valid_1's multi_logloss: 0.378472
[200]	training's multi_logloss: 0.266566	valid_1's multi_logloss: 0.351538
[300]	training's multi_logloss: 0.236617	valid_1's multi_logloss: 0.351732
Early stopping, best iteration is:
[241]	training's multi_logloss: 0.25268	valid_1's multi_logloss: 0.350652
Training until validation scores don't improve for 100 rounds.
[100]	training's multi_logloss: 0.333704	valid_1's multi_logloss: 0.369709
[200]	training's multi_logloss: 0.266637	valid_1's multi_logloss: 0.345884
[300]	training's multi_logloss: 0.236863	valid_1's multi_logloss: 0.34568
Early stopping, best iteration is:
[245]	training's multi_logloss: 0.251367	valid_1's multi_logloss: 0.344908
Training until validation scores don't improve for 100 rounds.
[100]	training's multi_logloss: 0.333918	valid_1's multi_logloss: 0.371349
[200]	training's multi_logloss: 0.266672	valid_1's multi_logloss: 0.


CPU times: user 3h 43min 18s, sys: 42.2 s, total: 3h 44min 1s
Wall time: 1h 1min 2s


In [ ]:
%%time
X = train.copy()
X.reset_index(drop = True, inplace = True)
y = X.target
X = X.drop(['target', 'ID'], axis = 1)

tes = test.drop('ID', axis = 1)

cat_seeds_preds = []
for i in tqdm_notebook(range(10)):
  X, y = shuffle(X,y, random_state = i)
  cat =  CatBoostClassifier(verbose = False, random_seed=i, task_type = 'GPU')
  cat.fit(X, y)
  cat_seeds_preds.append(cat.predict_proba(tes))

cat_seeds_sub = submission_preparation(cat_seeds_preds)


CPU times: user 5min 49s, sys: 3min 13s, total: 9min 2s
Wall time: 5min 20s


In [ ]:
blend = cat_stratified_sub.Label*0.45 + lgbm_grp_sub.Label*0.45 + cat_seeds_sub.Label *0.1
blend_sub = pd.DataFrame({'ID X PCODE': lgbm_grp_sub['ID X PCODE'], 'Label': blend})
blend_sub.to_csv('blend_unscaled_15fs_2.csv', index = False)